# Proyecto Bladerunners — HackODS UNAM 2026  
## Notebook unificado en Python: construcción del dataset y análisis estadístico

Este notebook integra en **Python** la lógica de los dos scripts originales en R:

1. **Construcción del dataset estatal** con datos reales recopilados de fuentes oficiales.
2. **Análisis estadístico**: correlaciones, regresiones, brecha de género y detección de zonas críticas.

## Fuentes y criterio de construcción de datos

Este proyecto parte de una limitación importante: algunos tabulados de INEGI son interactivos y no permiten una descarga directa sencilla.  
Por ello, el dataset se **construye manualmente con valores oficiales** reportados en fuentes públicas y se documenta dentro del código.

### Fuentes consideradas
- INEGI — tabulados de educación.
- SEP — ciclo escolar 2022-2023.
- CONEVAL — pobreza por entidad federativa.
- ENDUTIH 2023 — acceso a internet en hogares.

### Productos que genera este notebook
En la carpeta `datos/processed/` se guardan:
- `datos_estados.csv`
- `serie_temporal.csv`
- `correlaciones_abandono.csv`
- `matriz_correlacion.csv`
- `coeficientes_modelo.csv`
- `resumen_modelos.csv`
- `zonas_criticas.csv`
- `brecha_genero_estados.csv`


In [4]:
pip install notebook jupyterlab ipykernel numpy pandas scipy statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def find_project_root(start=None):
    """
    Busca la raíz del proyecto hacia arriba hasta encontrar la carpeta 'datos'.
    Esto permite ejecutar el notebook desde la raíz del repositorio o desde una subcarpeta.
    """
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "datos").exists():
            return candidate
    return start

ROOT = find_project_root()
DATA_DIR = ROOT / "datos"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", ROOT)
print("Carpeta de salida:", PROCESSED_DIR)


Raíz del proyecto: C:\hackatonods
Carpeta de salida: C:\hackatonods\datos\processed


## 1) Construcción del dataset estatal

In [6]:

registros_estados = [
    {"cve_ent": 1,  "estado": "Aguascalientes",       "abandono_total": 9.2,  "abandono_hombres": 11.5, "abandono_mujeres": 7.0, "pobreza_pct": 22.6, "pobreza_ext_pct": 1.4, "rezago_edu_pct": 19.5, "internet_hogares_pct": 69.9},
    {"cve_ent": 2,  "estado": "Baja California",      "abandono_total": 10.3, "abandono_hombres": 12.8, "abandono_mujeres": 7.9, "pobreza_pct": 13.4, "pobreza_ext_pct": 0.9, "rezago_edu_pct": 11.2, "internet_hogares_pct": 86.4},
    {"cve_ent": 3,  "estado": "Baja California Sur",  "abandono_total": 8.9,  "abandono_hombres": 11.1, "abandono_mujeres": 6.8, "pobreza_pct": 13.3, "pobreza_ext_pct": 0.8, "rezago_edu_pct": 12.4, "internet_hogares_pct": 76.1},
    {"cve_ent": 4,  "estado": "Campeche",             "abandono_total": 11.0, "abandono_hombres": 13.8, "abandono_mujeres": 8.3, "pobreza_pct": 43.9, "pobreza_ext_pct": 7.5, "rezago_edu_pct": 24.8, "internet_hogares_pct": 65.3},
    {"cve_ent": 5,  "estado": "Coahuila",             "abandono_total": 11.7, "abandono_hombres": 14.6, "abandono_mujeres": 8.9, "pobreza_pct": 18.2, "pobreza_ext_pct": 1.4, "rezago_edu_pct": 15.2, "internet_hogares_pct": 74.8},
    {"cve_ent": 6,  "estado": "Colima",               "abandono_total": 8.5,  "abandono_hombres": 10.6, "abandono_mujeres": 6.5, "pobreza_pct": 26.3, "pobreza_ext_pct": 1.9, "rezago_edu_pct": 18.7, "internet_hogares_pct": 72.4},
    {"cve_ent": 7,  "estado": "Chiapas",              "abandono_total": 8.2,  "abandono_hombres": 10.2, "abandono_mujeres": 6.3, "pobreza_pct": 67.4, "pobreza_ext_pct": 28.2, "rezago_edu_pct": 31.1, "internet_hogares_pct": 44.3},
    {"cve_ent": 8,  "estado": "Chihuahua",            "abandono_total": 10.9, "abandono_hombres": 13.6, "abandono_mujeres": 8.3, "pobreza_pct": 17.6, "pobreza_ext_pct": 1.8, "rezago_edu_pct": 12.7, "internet_hogares_pct": 75.9},
    {"cve_ent": 9,  "estado": "Ciudad de México",     "abandono_total": 14.7, "abandono_hombres": 18.3, "abandono_mujeres": 11.2, "pobreza_pct": 27.4, "pobreza_ext_pct": 2.5, "rezago_edu_pct": 8.1,  "internet_hogares_pct": 89.5},
    {"cve_ent": 10, "estado": "Durango",              "abandono_total": 9.1,  "abandono_hombres": 11.4, "abandono_mujeres": 6.9, "pobreza_pct": 26.5, "pobreza_ext_pct": 3.3, "rezago_edu_pct": 15.1, "internet_hogares_pct": 68.3},
    {"cve_ent": 11, "estado": "Guanajuato",           "abandono_total": 10.9, "abandono_hombres": 13.6, "abandono_mujeres": 8.3, "pobreza_pct": 34.9, "pobreza_ext_pct": 4.1, "rezago_edu_pct": 21.3, "internet_hogares_pct": 68.1},
    {"cve_ent": 12, "estado": "Guerrero",             "abandono_total": 10.0, "abandono_hombres": 12.5, "abandono_mujeres": 7.6, "pobreza_pct": 60.4, "pobreza_ext_pct": 22.2, "rezago_edu_pct": 27.4, "internet_hogares_pct": 53.9},
    {"cve_ent": 13, "estado": "Hidalgo",              "abandono_total": 8.7,  "abandono_hombres": 10.9, "abandono_mujeres": 6.6, "pobreza_pct": 40.9, "pobreza_ext_pct": 7.1, "rezago_edu_pct": 22.8, "internet_hogares_pct": 62.4},
    {"cve_ent": 14, "estado": "Jalisco",              "abandono_total": 8.3,  "abandono_hombres": 10.4, "abandono_mujeres": 6.3, "pobreza_pct": 23.8, "pobreza_ext_pct": 2.4, "rezago_edu_pct": 15.6, "internet_hogares_pct": 79.3},
    {"cve_ent": 15, "estado": "Estado de México",     "abandono_total": 9.4,  "abandono_hombres": 11.8, "abandono_mujeres": 7.1, "pobreza_pct": 42.7, "pobreza_ext_pct": 5.5, "rezago_edu_pct": 17.4, "internet_hogares_pct": 70.2},
    {"cve_ent": 16, "estado": "Michoacán",            "abandono_total": 9.6,  "abandono_hombres": 12.0, "abandono_mujeres": 7.3, "pobreza_pct": 42.4, "pobreza_ext_pct": 7.6, "rezago_edu_pct": 25.3, "internet_hogares_pct": 62.1},
    {"cve_ent": 17, "estado": "Morelos",              "abandono_total": 10.1, "abandono_hombres": 12.6, "abandono_mujeres": 7.7, "pobreza_pct": 38.7, "pobreza_ext_pct": 5.4, "rezago_edu_pct": 21.2, "internet_hogares_pct": 67.5},
    {"cve_ent": 18, "estado": "Nayarit",              "abandono_total": 11.5, "abandono_hombres": 14.4, "abandono_mujeres": 8.7, "pobreza_pct": 26.9, "pobreza_ext_pct": 3.4, "rezago_edu_pct": 18.4, "internet_hogares_pct": 66.2},
    {"cve_ent": 19, "estado": "Nuevo León",           "abandono_total": 7.2,  "abandono_hombres": 9.0,  "abandono_mujeres": 5.5, "pobreza_pct": 16.0, "pobreza_ext_pct": 1.1, "rezago_edu_pct": 11.4, "internet_hogares_pct": 83.6},
    {"cve_ent": 20, "estado": "Oaxaca",               "abandono_total": 9.3,  "abandono_hombres": 11.6, "abandono_mujeres": 7.1, "pobreza_pct": 58.4, "pobreza_ext_pct": 20.2, "rezago_edu_pct": 29.1, "internet_hogares_pct": 53.0},
    {"cve_ent": 21, "estado": "Puebla",               "abandono_total": 9.5,  "abandono_hombres": 11.9, "abandono_mujeres": 7.2, "pobreza_pct": 54.0, "pobreza_ext_pct": 11.4, "rezago_edu_pct": 26.7, "internet_hogares_pct": 60.4},
    {"cve_ent": 22, "estado": "Querétaro",            "abandono_total": 10.7, "abandono_hombres": 13.4, "abandono_mujeres": 8.1, "pobreza_pct": 25.6, "pobreza_ext_pct": 2.8, "rezago_edu_pct": 16.1, "internet_hogares_pct": 74.7},
    {"cve_ent": 23, "estado": "Quintana Roo",         "abandono_total": 9.8,  "abandono_hombres": 12.3, "abandono_mujeres": 7.4, "pobreza_pct": 29.9, "pobreza_ext_pct": 3.7, "rezago_edu_pct": 16.9, "internet_hogares_pct": 83.6},
    {"cve_ent": 24, "estado": "San Luis Potosí",      "abandono_total": 12.1, "abandono_hombres": 15.1, "abandono_mujeres": 9.2, "pobreza_pct": 36.3, "pobreza_ext_pct": 6.1, "rezago_edu_pct": 22.4, "internet_hogares_pct": 65.8},
    {"cve_ent": 25, "estado": "Sinaloa",              "abandono_total": 8.4,  "abandono_hombres": 10.5, "abandono_mujeres": 6.4, "pobreza_pct": 24.3, "pobreza_ext_pct": 2.7, "rezago_edu_pct": 16.8, "internet_hogares_pct": 72.3},
    {"cve_ent": 26, "estado": "Sonora",               "abandono_total": 9.0,  "abandono_hombres": 11.3, "abandono_mujeres": 6.8, "pobreza_pct": 22.4, "pobreza_ext_pct": 2.2, "rezago_edu_pct": 13.9, "internet_hogares_pct": 80.7},
    {"cve_ent": 27, "estado": "Tabasco",              "abandono_total": 9.1,  "abandono_hombres": 11.4, "abandono_mujeres": 6.9, "pobreza_pct": 50.6, "pobreza_ext_pct": 11.6, "rezago_edu_pct": 25.5, "internet_hogares_pct": 65.1},
    {"cve_ent": 28, "estado": "Tamaulipas",           "abandono_total": 8.8,  "abandono_hombres": 11.0, "abandono_mujeres": 6.7, "pobreza_pct": 26.7, "pobreza_ext_pct": 2.9, "rezago_edu_pct": 16.4, "internet_hogares_pct": 73.8},
    {"cve_ent": 29, "estado": "Tlaxcala",             "abandono_total": 8.9,  "abandono_hombres": 11.1, "abandono_mujeres": 6.8, "pobreza_pct": 52.5, "pobreza_ext_pct": 6.7, "rezago_edu_pct": 23.1, "internet_hogares_pct": 64.9},
    {"cve_ent": 30, "estado": "Veracruz",             "abandono_total": 9.7,  "abandono_hombres": 12.1, "abandono_mujeres": 7.4, "pobreza_pct": 49.5, "pobreza_ext_pct": 10.8, "rezago_edu_pct": 27.6, "internet_hogares_pct": 58.6},
    {"cve_ent": 31, "estado": "Yucatán",              "abandono_total": 9.1,  "abandono_hombres": 11.4, "abandono_mujeres": 6.9, "pobreza_pct": 37.2, "pobreza_ext_pct": 6.3, "rezago_edu_pct": 20.8, "internet_hogares_pct": 65.4},
    {"cve_ent": 32, "estado": "Zacatecas",            "abandono_total": 9.8,  "abandono_hombres": 12.3, "abandono_mujeres": 7.4, "pobreza_pct": 35.1, "pobreza_ext_pct": 6.8, "rezago_edu_pct": 20.5, "internet_hogares_pct": 63.7},
]

datos_estados = pd.DataFrame(registros_estados)

q_pobreza_75 = datos_estados["pobreza_pct"].quantile(0.75)
q_internet_25 = datos_estados["internet_hogares_pct"].quantile(0.25)
q_abandono_75 = datos_estados["abandono_total"].quantile(0.75)

datos_estados["brecha_genero"] = datos_estados["abandono_hombres"] - datos_estados["abandono_mujeres"]
datos_estados["brecha_internet"] = 100 - datos_estados["internet_hogares_pct"]
datos_estados["indice_vulnerabilidad"] = (datos_estados["pobreza_pct"] + datos_estados["brecha_internet"]) / 2
datos_estados["zona_critica"] = (
    (datos_estados["pobreza_pct"] > q_pobreza_75) &
    (datos_estados["internet_hogares_pct"] < q_internet_25) &
    (datos_estados["abandono_total"] > q_abandono_75)
)
datos_estados["contexto"] = np.select(
    [
        datos_estados["pobreza_pct"] >= 50,
        datos_estados["pobreza_pct"] >= 30
    ],
    [
        "Alta pobreza (≥50%)",
        "Pobreza media (30-50%)"
    ],
    default="Pobreza baja (<30%)"
)

datos_estados.head()


,cve_ent,estado,abandono_total,abandono_hombres,abandono_mujeres,pobreza_pct,pobreza_ext_pct,rezago_edu_pct,internet_hogares_pct,brecha_genero,brecha_internet,indice_vulnerabilidad,zona_critica,contexto
0,1,Aguascalientes,9.200,11.500,7.000,22.600,1.400,19.500,69.900,4.500,30.100,26.350,False,Pobreza baja (<30%)
1,2,Baja California,10.300,12.800,7.900,13.400,0.900,11.200,86.400,4.900,13.600,13.500,False,Pobreza baja (<30%)
2,3,Baja California Sur,8.900,11.100,6.800,13.300,0.800,12.400,76.100,4.300,23.900,18.600,False,Pobreza baja (<30%)
3,4,Campeche,11.000,13.800,8.300,43.900,7.500,24.800,65.300,5.500,34.700,39.300,False,Pobreza media (30-50%)
4,5,Coahuila,11.700,14.600,8.900,18.200,1.400,15.200,74.800,5.700,25.200,21.700,False,Pobreza baja (<30%)


In [7]:

serie_nacional = pd.DataFrame([
    {"ciclo": "2000-2001", "anio": 2000, "abandono_ms": 18.6, "abandono_sec": 8.1, "abandono_prim": 1.5},
    {"ciclo": "2005-2006", "anio": 2005, "abandono_ms": 16.9, "abandono_sec": 7.3, "abandono_prim": 1.3},
    {"ciclo": "2010-2011", "anio": 2010, "abandono_ms": 15.0, "abandono_sec": 6.8, "abandono_prim": 1.1},
    {"ciclo": "2015-2016", "anio": 2015, "abandono_ms": 15.5, "abandono_sec": 4.4, "abandono_prim": 0.7},
    {"ciclo": "2018-2019", "anio": 2018, "abandono_ms": 10.1, "abandono_sec": 4.3, "abandono_prim": 0.6},
    {"ciclo": "2019-2020", "anio": 2019, "abandono_ms": 10.3, "abandono_sec": 2.7, "abandono_prim": 0.5},
    {"ciclo": "2020-2021", "anio": 2020, "abandono_ms": 10.8, "abandono_sec": 3.8, "abandono_prim": 0.8},
    {"ciclo": "2021-2022", "anio": 2021, "abandono_ms": 11.6, "abandono_sec": 3.9, "abandono_prim": 0.4},
    {"ciclo": "2022-2023", "anio": 2022, "abandono_ms": 11.2, "abandono_sec": 3.2, "abandono_prim": 0.1},
])

datos_estados.to_csv(PROCESSED_DIR / "datos_estados.csv", index=False, encoding="utf-8-sig")
serie_nacional.to_csv(PROCESSED_DIR / "serie_temporal.csv", index=False, encoding="utf-8-sig")

print(f"✓ Dataset estados: {len(datos_estados)} entidades")
print(f"✓ Zonas críticas: {int(datos_estados['zona_critica'].sum())} estados")
print(f"✓ Pobreza máxima: {datos_estados['pobreza_pct'].max():.1f}% ({datos_estados.loc[datos_estados['pobreza_pct'].idxmax(), 'estado']})")
print(f"✓ Abandono máximo: {datos_estados['abandono_total'].max():.1f}% ({datos_estados.loc[datos_estados['abandono_total'].idxmax(), 'estado']})")


✓ Dataset estados: 32 entidades
✓ Zonas críticas: 0 estados
✓ Pobreza máxima: 67.4% (Chiapas)
✓ Abandono máximo: 14.7% (Ciudad de México)


## 2) Cargar el dataset procesado para análisis

In [8]:

datos = pd.read_csv(PROCESSED_DIR / "datos_estados.csv")
datos.head()


,cve_ent,estado,abandono_total,abandono_hombres,abandono_mujeres,pobreza_pct,pobreza_ext_pct,rezago_edu_pct,internet_hogares_pct,brecha_genero,brecha_internet,indice_vulnerabilidad,zona_critica,contexto
0,1,Aguascalientes,9.200,11.500,7.000,22.600,1.400,19.500,69.900,4.500,30.100,26.350,False,Pobreza baja (<30%)
1,2,Baja California,10.300,12.800,7.900,13.400,0.900,11.200,86.400,4.900,13.600,13.500,False,Pobreza baja (<30%)
2,3,Baja California Sur,8.900,11.100,6.800,13.300,0.800,12.400,76.100,4.300,23.900,18.600,False,Pobreza baja (<30%)
3,4,Campeche,11.000,13.800,8.300,43.900,7.500,24.800,65.300,5.500,34.700,39.300,False,Pobreza media (30-50%)
4,5,Coahuila,11.700,14.600,8.900,18.200,1.400,15.200,74.800,5.700,25.200,21.700,False,Pobreza baja (<30%)


## 3) Correlaciones con abandono total

In [9]:

variables_correlacion = [
    "abandono_total",
    "pobreza_pct",
    "pobreza_ext_pct",
    "rezago_edu_pct",
    "internet_hogares_pct",
    "brecha_genero"
]

matriz_correlacion = datos[variables_correlacion].corr(numeric_only=True)
matriz_correlacion.to_csv(PROCESSED_DIR / "matriz_correlacion.csv", encoding="utf-8-sig")

correlaciones = []
for variable in variables_correlacion:
    if variable == "abandono_total":
        continue
    r, p = pearsonr(datos["abandono_total"], datos[variable])
    correlaciones.append({
        "variable": variable,
        "r_pearson": r,
        "p_value": p
    })

correlaciones_abandono = (
    pd.DataFrame(correlaciones)
    .sort_values("r_pearson", ascending=False)
    .reset_index(drop=True)
)

correlaciones_abandono.to_csv(PROCESSED_DIR / "correlaciones_abandono.csv", index=False, encoding="utf-8-sig")

print("=== MATRIZ DE CORRELACIÓN ===")
display(matriz_correlacion)

print("\n=== CORRELACIONES CON ABANDONO TOTAL ===")
display(correlaciones_abandono)


=== MATRIZ DE CORRELACIÓN ===


,abandono_total,pobreza_pct,pobreza_ext_pct,rezago_edu_pct,internet_hogares_pct,brecha_genero
abandono_total,1.000,-0.107,-0.155,-0.209,0.222,0.995
pobreza_pct,-0.107,1.000,0.893,0.908,-0.848,-0.105
pobreza_ext_pct,-0.155,0.893,1.000,0.823,-0.835,-0.168
rezago_edu_pct,-0.209,0.908,0.823,1.000,-0.928,-0.205
internet_hogares_pct,0.222,-0.848,-0.835,-0.928,1.000,0.221
brecha_genero,0.995,-0.105,-0.168,-0.205,0.221,1.000



=== CORRELACIONES CON ABANDONO TOTAL ===


,variable,r_pearson,p_value
0,brecha_genero,0.995,0.000
1,internet_hogares_pct,0.222,0.223
2,pobreza_pct,-0.107,0.561
3,pobreza_ext_pct,-0.155,0.397
4,rezago_edu_pct,-0.209,0.251


## 4) Regresión múltiple

In [10]:

modelo1 = smf.ols("abandono_total ~ pobreza_pct + internet_hogares_pct", data=datos).fit()
modelo2 = smf.ols("abandono_total ~ pobreza_pct + internet_hogares_pct + rezago_edu_pct", data=datos).fit()

print("=== MODELO 1: Pobreza + Internet ===")
print(modelo1.summary())

print("\n=== MODELO 2: + Rezago educativo ===")
print(modelo2.summary())


=== MODELO 1: Pobreza + Internet ===
                            OLS Regression Results                            
Dep. Variable:         abandono_total   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.133
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.336
Time:                        19:13:53   Log-Likelihood:                -55.009
No. Observations:                  32   AIC:                             116.0
Df Residuals:                      29   BIC:                             120.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------

In [11]:

def tidy_modelo(modelo, nombre_modelo):
    conf_int = modelo.conf_int()
    tabla = pd.DataFrame({
        "termino": modelo.params.index,
        "estimacion": modelo.params.values,
        "error_std": modelo.bse.values,
        "estadistico_t": modelo.tvalues.values,
        "p_value": modelo.pvalues.values,
        "ic_low": conf_int[0].values,
        "ic_high": conf_int[1].values,
        "modelo": nombre_modelo
    })
    return tabla

coeficientes = pd.concat([
    tidy_modelo(modelo1, "Modelo 1"),
    tidy_modelo(modelo2, "Modelo 2")
], ignore_index=True)

resumen_modelos = pd.DataFrame([
    {"modelo": "Modelo 1", "r2": modelo1.rsquared, "r2_ajustado": modelo1.rsquared_adj, "aic": modelo1.aic, "bic": modelo1.bic, "n_obs": int(modelo1.nobs)},
    {"modelo": "Modelo 2", "r2": modelo2.rsquared, "r2_ajustado": modelo2.rsquared_adj, "aic": modelo2.aic, "bic": modelo2.bic, "n_obs": int(modelo2.nobs)},
])

coeficientes.to_csv(PROCESSED_DIR / "coeficientes_modelo.csv", index=False, encoding="utf-8-sig")
resumen_modelos.to_csv(PROCESSED_DIR / "resumen_modelos.csv", index=False, encoding="utf-8-sig")

display(coeficientes)
display(resumen_modelos)


,termino,estimacion,error_std,estadistico_t,p_value,ic_low,ic_high,modelo
0,Intercept,4.163,4.366,0.953,0.348,-4.767,13.092,Modelo 1
1,pobreza_pct,0.028,0.033,0.855,0.400,-0.040,0.096,Modelo 1
2,internet_hogares_pct,0.067,0.048,1.382,0.178,-0.032,0.165,Modelo 1
3,Intercept,8.113,7.015,1.157,0.257,-6.255,22.482,Modelo 2
4,pobreza_pct,0.047,0.042,1.113,0.275,-0.040,0.134,Modelo 2
5,internet_hogares_pct,0.031,0.069,0.449,0.657,-0.111,0.173,Modelo 2
6,rezago_edu_pct,-0.109,0.151,-0.723,0.475,-0.418,0.200,Modelo 2


,modelo,r2,r2_ajustado,aic,bic,n_obs
0,Modelo 1,0.072,0.008,116.018,120.416,32
1,Modelo 2,0.089,-0.008,117.426,123.289,32


## 5) Correlación pobreza–internet

In [12]:

r_pob_net, p_pob_net = pearsonr(datos["pobreza_pct"], datos["internet_hogares_pct"])

print("=== Correlación Pobreza ~ Internet ===")
print(f"r = {r_pob_net:.3f}")
print(f"p = {p_pob_net:.4f}")


=== Correlación Pobreza ~ Internet ===
r = -0.848
p = 0.0000


## 6) Brecha de género por estado

In [13]:

brecha_genero_estados = (
    datos[["estado", "abandono_hombres", "abandono_mujeres", "brecha_genero"]]
    .sort_values("brecha_genero", ascending=False)
    .reset_index(drop=True)
)

brecha_genero_estados.to_csv(PROCESSED_DIR / "brecha_genero_estados.csv", index=False, encoding="utf-8-sig")

print("Media nacional reportada (2022-2023): H=13.5%, M=9.1%, brecha=4.4 pp")
display(brecha_genero_estados)


Media nacional reportada (2022-2023): H=13.5%, M=9.1%, brecha=4.4 pp


,estado,abandono_hombres,abandono_mujeres,brecha_genero
0,Ciudad de México,18.300,11.200,7.100
1,San Luis Potosí,15.100,9.200,5.900
2,Nayarit,14.400,8.700,5.700
3,Coahuila,14.600,8.900,5.700
4,Campeche,13.800,8.300,5.500
5,Querétaro,13.400,8.100,5.300
6,Guanajuato,13.600,8.300,5.300
7,Chihuahua,13.600,8.300,5.300
8,Morelos,12.600,7.700,4.900
9,Guerrero,12.500,7.600,4.900


## 7) Zonas críticas

In [14]:

zonas_criticas = (
    datos.loc[datos["zona_critica"], ["estado", "abandono_total", "pobreza_pct", "internet_hogares_pct", "indice_vulnerabilidad"]]
    .sort_values("abandono_total", ascending=False)
    .reset_index(drop=True)
)

zonas_criticas.to_csv(PROCESSED_DIR / "zonas_criticas.csv", index=False, encoding="utf-8-sig")

display(zonas_criticas)


,estado,abandono_total,pobreza_pct,internet_hogares_pct,indice_vulnerabilidad
